# Build and validate T-COMP-PT v1.0

This notebook can reacquire the three 32.6 MB source files from their
authoritative CC BY 4.0 repository, rebuild the normalized paired paths, and
rerun every frame-level QA check.

## Evidence boundary

The included 1,034 images and masks are experimental public data. They come
from one specimen and are therefore marked `benchmark_only`. The 6.56 GB
curved/planar/trapezoidal corpus is registered and fetched on demand. No
synthetic image is represented as an experiment.

In [ ]:
from pathlib import Path
import csv, json, shutil, subprocess, sys, urllib.request

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
SOURCE_CACHE = ROOT / "source_cache"
SOURCE_CACHE.mkdir(exist_ok=True)
BUILD = ROOT / "src" / "build_thermography_dataset.py"
VALIDATE = ROOT / "src" / "validate_thermography_dataset.py"
assert BUILD.exists() and VALIDATE.exists(), ROOT

In [ ]:
with (ROOT / "metadata" / "source_archives.csv").open(newline="") as f:
    archives = list(csv.DictReader(f))
for row in archives:
    target = SOURCE_CACHE / row["filename"]
    if not target.exists() or target.stat().st_size != int(row["expected_size_bytes"]):
        print("Downloading", row["filename"])
        request = urllib.request.Request(
            row["download_url"],
            headers={"User-Agent": "Mozilla/5.0", "Accept": "application/octet-stream"},
        )
        with urllib.request.urlopen(request) as response, target.open("wb") as out:
            shutil.copyfileobj(response, out)
print("Source cache ready:", SOURCE_CACHE)

In [ ]:
subprocess.run(
    [sys.executable, str(BUILD), "--source-dir", str(SOURCE_CACHE), "--output", str(ROOT)],
    check=True,
)
subprocess.run(
    [sys.executable, str(VALIDATE), "--dataset", str(ROOT)],
    check=True,
)
qa = json.loads((ROOT / "reports" / "qa_report.json").read_text())
qa

In [ ]:
from PIL import Image

image = Image.open(ROOT / "data/public_benchmark/images/frame_0500.png")
mask = Image.open(ROOT / "data/public_benchmark/masks/frame_0500.png")
print("Deposited image:", image.size, image.mode)
print("Deposited mask:", mask.size, mask.mode)
display(image)
display(mask)

## Publication gate

Frame-wise random splitting is prohibited. Generalization claims must use
independent specimens and batches, with both sides of each specimen kept in
one partition. For landing-gear claims, acquire the project-specific curved
specimens under a frozen protocol.